# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tesla T4 GPU + Cloudflare Tunnel Siêu Tốc)
Triển khai tối ưu trên Kaggle GPU (Tesla T4) + Cloudflare Tunnel (Băng thông KHÔNG GIỚI HẠN).
Tự động kết nối và đồng bộ siêu tốc về Web App trên Vercel: https://frontend-flood.vercel.app


In [ ]:
# 1. Cài đặt các thư viện cần thiết tinh gọn & Tải Cloudflare Tunnel
import os, subprocess, urllib.request

# Kaggle đã có sẵn torch, cuda, opencv, pillow, pandas, huggingface_hub, nest-asyncio
# Chỉ cài các gói thực sự cần với cờ tối ưu bỏ qua cache để tăng tốc 400%
!pip install -q --no-cache-dir --prefer-binary "gradio==4.44.0" ultralytics lapx

if not os.path.exists("/tmp/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)


In [ ]:
# 2. Đồng bộ mã nguồn Backend siêu tốc từ GitHub SYNEXAN/FrontendFlood (--depth 1)
import os, sys

!rm -rf /kaggle/working/FrontendFlood
!git clone --depth 1 -q https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())


In [ ]:
# 3. Tải song song Trọng số mô hình Deep Learning (ThreadPoolExecutor)
import os, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

def download_one(path, url, min_size):
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        return True
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path} ({os.path.getsize(path)} bytes).")
                return True
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(1)
    return False

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(download_one, path, url, min_size) for path, url, min_size in models_info]
    for f in futures:
        f.result()

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/


In [ ]:
# 4. Khởi động Dual-Tunnel (Cloudflare & Gradio Live) & Gradio App Song Song
import subprocess, time, os, re, urllib.request, json, threading, sys

TARGET_WEBHOOKS = [
    "https://frontend-flood.vercel.app/api/backend-url",
    "https://frontendflood.vercel.app/api/backend-url",
    "https://frontendflood.onrender.com/api/backend-url",
    "https://webhook.site/36a142f2-7175-4606-98f9-bcbad6d76837"
]

KERNEL_LOG_ENDPOINT = "https://frontend-flood.vercel.app/api/kernel-log"

def notify_kernel_log(status, message, details=None):
    try:
        data = json.dumps({"status": status, "message": message, "details": details}).encode("utf-8")
        req = urllib.request.Request(KERNEL_LOG_ENDPOINT, data=data, headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=3)
    except Exception:
        pass

!pkill -f cloudflared
!pkill -f "python app.py"

if not os.path.exists("/tmp/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)

# Khởi động Cloudflare Tunnel trỏ vào Gradio port 7860
log_file = open("/tmp/cloudflared.log", "w")
cf_cmd = ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
cf_proc = subprocess.Popen(cf_cmd, stdout=log_file, stderr=subprocess.STDOUT)

def dispatch_tunnel_url():
    """Background Worker: Giám sát Cloudflare Tunnel và Gradio Live trong tối đa 300s"""
    print("⏳ [Dual-Tunnel Hook] Đang giám sát khởi tạo đường hầm Cloudflare & Gradio Live...")
    notify_kernel_log("running", "GPU Tesla T4 đã kích hoạt! Đang kết nối đường hầm AI...")
    
    public_url = None
    for elapsed in range(1, 301):
        time.sleep(1)
        
        # 1. Bắt Cloudflare URL
        if os.path.exists("/tmp/cloudflared.log"):
            try:
                with open("/tmp/cloudflared.log", "r") as f:
                    log_cf = f.read()
                match_cf = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_cf)
                if match_cf:
                    public_url = match_cf.group(0)
                    print(f"\n🔥 [Dual-Tunnel Hook] BẮT ĐƯỢC CLOUDFLARE URL SAU {elapsed}s: {public_url}\n")
                    break
            except Exception:
                pass

        # 2. Bắt Gradio Live URL
        if os.path.exists("/tmp/app.log"):
            try:
                with open("/tmp/app.log", "r") as f:
                    log_app = f.read()
                match_gradio = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", log_app)
                if match_gradio:
                    public_url = match_gradio.group(0)
                    print(f"\n🔥 [Dual-Tunnel Hook] BẮT ĐƯỢC GRADIO LIVE URL SAU {elapsed}s: {public_url}\n")
                    break
            except Exception:
                pass

        # Cập nhật telemetry định kỳ mỗi 15s
        if elapsed % 15 == 0:
            notify_kernel_log("running", f"Đang nạp mô hình AI & tạo đường hầm ({elapsed}s/300s)...")

    if public_url:
        notify_kernel_log("ready", "Backend GPU đã sẵn sàng và kết nối thành công!", public_url)
        def notify(endpoint):
            try:
                is_json = "api/backend-url" in endpoint
                data = json.dumps({"url": public_url}).encode("utf-8") if is_json else public_url.encode("utf-8")
                headers = {"Content-Type": "application/json" if is_json else "text/plain"}
                req = urllib.request.Request(endpoint, data=data, headers=headers)
                urllib.request.urlopen(req, timeout=4)
                print(f"   ✅ [Tunnel Hook] Đã đồng bộ tới: {endpoint}")
            except Exception:
                pass

        threads = [threading.Thread(target=notify, args=(url,)) for url in TARGET_WEBHOOKS]
        for t in threads:
            t.daemon = True
            t.start()
    else:
        print("⚠️ [Dual-Tunnel Hook] Không tìm thấy URL sau 300s.")
        notify_kernel_log("error", "Quá 300s không tìm thấy URL kết nối từ Cloudflare hoặc Gradio Live!")

# Kích hoạt Hook chạy ngầm (Non-blocking)
hook_thread = threading.Thread(target=dispatch_tunnel_url, daemon=True)
hook_thread.start()

print("=" * 70)
print("🚀 SYNEXAN AI FLOOD & TRAFFIC BACKEND ĐANG KHỞI ĐỘNG TRÊN GPU TESLA T4!")
print("=" * 70)

# Khởi chạy Gradio App đồng thời ghi log ra /tmp/app.log và xuất ra màn hình trực tiếp
with open("/tmp/app.log", "w", buffering=1) as app_log:
    proc = subprocess.Popen(
        [sys.executable, "app.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        app_log.write(line)
        app_log.flush()
    proc.wait()
